# 10. EKF Localization

EKF localization은 지도가 알려져 있고, 로봇 pose만 Gaussian으로 추정한다.

$$x_t=[x,y,\theta]^T, \qquad bel(x_t)=\mathcal{N}(\mu_t,\Sigma_t)$$

Motion model과 landmark sensor model을 둘 다 Jacobian으로 선형화한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Landmark 기반 EKF Localization

로봇은 velocity command로 움직이고, 각 landmark에 대한 range-bearing 측정을 사용해 pose를 보정한다.

In [ ]:
np.random.seed(22)
def wrap(a): return np.arctan2(np.sin(a),np.cos(a))
landmarks=np.array([[2,2],[8,2],[8,7],[2,7]])
dt=0.1; steps=110
u=np.column_stack([np.ones(steps)*0.75, 0.35*np.sin(np.linspace(0,5,steps))])

def motion(x,u):
    v,w=u; th=x[2]
    return np.array([x[0]+v*dt*np.cos(th), x[1]+v*dt*np.sin(th), wrap(th+w*dt)])
def G_jac(x,u):
    v,w=u; th=x[2]
    return np.array([[1,0,-v*dt*np.sin(th)],[0,1,v*dt*np.cos(th)],[0,0,1]])
def V_jac(x,u):
    th=x[2]
    return np.array([[dt*np.cos(th),0],[dt*np.sin(th),0],[0,dt]])
def h(x,m):
    dx=m[0]-x[0]; dy=m[1]-x[1]
    return np.array([np.hypot(dx,dy), wrap(np.arctan2(dy,dx)-x[2])])
def H_jac(x,m):
    dx=m[0]-x[0]; dy=m[1]-x[1]; q=dx*dx+dy*dy; r=np.sqrt(q)
    return np.array([[-dx/r,-dy/r,0],[dy/q,-dx/q,-1]])

true=np.array([1.0,1.0,0.2])
mu=np.array([0.7,0.8,0.0]); P=np.diag([0.4,0.4,0.2])
M_noise=np.diag([0.06**2,0.04**2]); Q=np.diag([0.15**2,np.deg2rad(4)**2])
truth=[]; est=[]; sig=[]
for k in range(steps):
    true=motion(true,u[k]+np.random.multivariate_normal([0,0],M_noise))
    # predict
    G=G_jac(mu,u[k]); V=V_jac(mu,u[k])
    mu=motion(mu,u[k]); P=G@P@G.T + V@M_noise@V.T
    # update all visible landmarks
    for m in landmarks:
        z=h(true,m)+np.random.multivariate_normal([0,0],Q)
        z[1]=wrap(z[1])
        zhat=h(mu,m); Hm=H_jac(mu,m)
        innov=z-zhat; innov[1]=wrap(innov[1])
        S=Hm@P@Hm.T+Q; K=P@Hm.T@np.linalg.inv(S)
        mu=mu+K@innov; mu[2]=wrap(mu[2])
        P=(np.eye(3)-K@Hm)@P
    truth.append(true.copy()); est.append(mu.copy()); sig.append(np.sqrt(np.diag(P)))
truth=np.array(truth); est=np.array(est); sig=np.array(sig)

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(truth[:,0],truth[:,1],'k-',lw=2,label='true')
axes[0].plot(est[:,0],est[:,1],color='#E85D24',lw=2,label='EKF estimate')
axes[0].scatter(landmarks[:,0],landmarks[:,1],marker='*',s=160,color='#1D9E75',label='landmarks')
axes[0].axis('equal'); axes[0].grid(alpha=0.25); axes[0].legend(); axes[0].set_title('EKF localization trajectory')
axes[1].plot(sig[:,0],label='sigma x'); axes[1].plot(sig[:,1],label='sigma y'); axes[1].plot(np.rad2deg(sig[:,2]),label='sigma theta deg')
axes[1].grid(alpha=0.25); axes[1].legend(); axes[1].set_title('posterior uncertainty')
plt.tight_layout(); plt.savefig('assets/10_ekf_localization.png',dpi=150,bbox_inches='tight'); plt.show()
print('final position error:', np.linalg.norm(est[-1,:2]-truth[-1,:2]).round(4))

## 요약

| 단계 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| EKF prediction | 비선형 motion model 선형화 | Ch.7 Gaussian localization |
| EKF correction | landmark measurement 선형화 | Ch.6 perception + Ch.7 localization |
| Known map | landmark 위치는 고정 | localization과 SLAM의 차이 |